# Use classic way to calibrate allsky camera


In [ ]:

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt



In [ ]:
# load data from file
import pandas as pd

df = pd.read_csv('pairs.csv')
azalt_original = df[['az', 'alt']].to_numpy()
photo_original = df[['extracted_x', 'extracted_y']].to_numpy()

photo_original

In [ ]:
azalt_original

## DATA prepare | altaz to polar

In [ ]:
def azalt_to_polar(azalt):
    rho = np.deg2rad(90 - azalt[..., 1]) / (np.pi / 2)
    rho *= 983
    theta = np.deg2rad(azalt[..., 0])
    theta += np.deg2rad(155.6)  # initial rotation angle
    theta %= 2 * np.pi
    return np.array([rho, theta]).T


polar = azalt_to_polar(azalt_original)


def photo_to_polar(photo):
    photo = photo - [1053, 1063]  # initial center point
    rho = np.sqrt(photo[..., 0] ** 2 + photo[..., 1] ** 2)
    theta = np.arctan2(photo[..., 1], photo[..., 0]) % (2 * np.pi)

    return np.array([rho, theta]).T


photo_polar = photo_to_polar(photo_original)

In [ ]:
def plot_polar(photo_polar, polar):
    ax1 = plt.subplot(121, polar=True)
    ax1.set_theta_direction(-1)
    sns.scatterplot(x=photo_polar[:, 1], y=photo_polar[:, 0])
    ax2 = plt.subplot(122, polar=True)
    ax2.set_theta_direction(-1)
    sns.scatterplot(x=polar[:, 1], y=polar[:, 0], color='red')


plot_polar(photo_polar, polar)

In [ ]:
photo_polar.max(axis=0), photo_polar.min(axis=0)

In [ ]:
polar.max(axis=0), polar.min(axis=0)

In [ ]:
error = photo_polar - polar
print(error.argmin(axis=0), error.max(axis=0))
sns.jointplot(x=error[:, 0], y=error[:, 1])
plt.xlabel('rho error')
plt.ylabel('theta error')
print(error.min(axis=0), error.max(axis=0))

## polar projection


In [ ]:
from scipy.optimize import leastsq, curve_fit


def rho_projection(rho, k):
    tmp_rho = np.array([rho, rho ** 3, rho ** 5, rho ** 7, rho ** 9]).T
    print(tmp_rho)
    return np.dot(tmp_rho, k)


def rho_projection_k(rho, k0, k1, k2, k3, k4):
    result = k0 * rho
    result += k1 * rho ** 3
    result += k2 * rho ** 5
    result += k3 * rho ** 7
    result += k4 * rho ** 9

    return result

# k = np.array([1, 2, 3, 4, 5])
# rho_1 = rho_projection(k, polar[:, 0])
# 
# polar_1 = polar.copy()
# polar_1[:, 0] = rho_1

In [ ]:

popt, pcov = curve_fit(rho_projection_k, xdata=polar[:, 0], ydata=photo_polar[:, 0])
proj_param = popt
proj_param


In [ ]:
polar1 = polar.copy()
polar1[:, 0] = rho_projection_k(polar[:, 0], *proj_param)

# draw scatter plot in photo coordinate
plot_polar(photo_polar, polar1)
plt.show()

In [ ]:
error1 = photo_polar - polar1
sns.jointplot(x=error1[:, 0], y=error1[:, 1])
plt.xlabel('rho error')
plt.ylabel('theta error')
print(error1.min(axis=0), error1.max(axis=0))
print('mean error', error1.mean(axis=0))

## radial distortion and tangential distortion

In [ ]:
def distortion(polar, radial, tangential):
    rho = polar[..., 0]
    theta = polar[..., 1]

    tmp_rho = np.array([rho, rho ** 3, rho ** 5]).T
    radial_term = tmp_rho.dot(radial)

    tmp_theta = np.array([np.cos(theta), np.sin(theta),
                          np.cos(2 * theta), np.sin(2 * theta)]).T
    tangential_term = tmp_theta.dot(tangential)

    # print(tmp_rho, radial_term)
    # print(tmp_theta, tangential_term)
    result = radial_term * tangential_term

    return result


def distorion_rho(polar, radial, tangential):
    rho = polar[..., 0]
    result = rho + distortion(polar, radial, tangential)
    return result


def distorion_theta(polar, radial, tangential):
    theta = polar[..., 1]
    result = theta + distortion(polar, radial, tangential)
    result %= 2 * np.pi
    return result

# radial = np.array([1, 1, 1])
# tangential = np.array([2, 2, 2, 2])
# 
# d0=distortion(polar1, radial, tangential)


In [ ]:
def distortion_lm(polar,
                  radial0, radial1, radial2,
                  tan0, tan1, tan2, tan3, ):
    rho = polar[..., 0]
    theta = polar[..., 1]

    tmp_rho = np.array([rho, rho ** 3, rho ** 5]).T
    radial_term = tmp_rho.dot([radial0, radial1, radial2])

    tmp_theta = np.array([np.cos(theta), np.sin(theta),
                          np.cos(2 * theta), np.sin(2 * theta)]).T
    tangential_term = tmp_theta.dot([tan0, tan1, tan2, tan3])

    result = radial_term * tangential_term
    return result


def distortion_lm_rho(polar,
                      radial0, radial1, radial2,
                      tan0, tan1, tan2, tan3):
    rho = polar[..., 0]
    result = rho + distortion_lm(polar, radial0, radial1, radial2, tan0, tan1, tan2, tan3)
    return result


def distortion_lm_theta(polar,
                        radial0, radial1, radial2,
                        tan0, tan1, tan2, tan3):
    theta = polar[..., 1]
    result = theta + distortion_lm(polar, radial0, radial1, radial2, tan0, tan1, tan2, tan3)
    return result


radial = np.array([1, 1, 1, ])
tangential = np.array([2, 2, 2, 2, ])

d = distortion_lm(polar1,
                  radial[0], radial[1], radial[2],
                  tangential[0], tangential[1], tangential[2], tangential[3])

In [ ]:
popt, pcov = curve_fit(distortion_lm_rho, xdata=polar, ydata=photo_polar[..., 0])
lm_pred_rho = popt
lm_pred_rho

In [ ]:
popt, pcov = curve_fit(distortion_lm_theta, xdata=polar, ydata=photo_polar[..., 1])
lm_pred_theta = popt
lm_pred_theta

In [ ]:
polar2 = polar1.copy()
polar2[:, 0] = distortion_lm_rho(polar1, *lm_pred_rho)
polar2[:, 1] = distortion_lm_theta(polar1, *lm_pred_theta)
polar2

In [ ]:
plot_polar(photo_polar, polar2)

In [ ]:
error2 = photo_polar - polar2
sns.jointplot(x=error2[:, 0], y=error2[:, 1])
plt.xlabel('rho error')
plt.ylabel('theta error')

print(error2.min(axis=0), error2.max(axis=0))
print('mean error', error2.mean(axis=0))

## Overall curve_fit


In [ ]:
def overall_rho(polar, k0, k1, k2, k3, k4, r0, r1, r2, t0, t1, t2, t3):
    rho1 = rho_projection_k(polar[:, 0], k0, k1, k2, k3, k4)
    rho2 = distortion_lm_rho(polar, r0, r1, r2, t0, t1, t2, t3)
    return rho1 + rho2


def overall_theta(polar, r0, r1, r2, t0, t1, t2, t3):
    theta2 = distortion_lm_theta(polar, r0, r1, r2, t0, t1, t2, t3)
    return theta2


popt, pcov = curve_fit(overall_rho, xdata=polar, ydata=photo_polar[..., 0],
                       p0=[*proj_param, *lm_pred_rho])
rho_pram = popt
popt, pcov = curve_fit(overall_theta, xdata=polar, ydata=photo_polar[..., 1],
                       p0=[*lm_pred_theta])
theta_pram = popt


In [ ]:
rho_pram

In [ ]:
theta_pram

In [ ]:
polar3 = polar.copy()
polar3[:, 0] = overall_rho(polar, *rho_pram)
polar3[:, 1] = overall_theta(polar, *theta_pram)

plot_polar(photo_polar, polar3)

In [ ]:
error3 = photo_polar - polar3
sns.jointplot(x=error3[:, 0], y=error3[:, 1])
plt.xlabel('rho error')
plt.ylabel('theta error')

print(error3.min(axis=0), error3.max(axis=0), np.abs(error3).mean(axis=0))

In [ ]:
error3.argmax(axis=0)

In [ ]:
def polar_to_photo(polar, centerX, centerY):
    x = polar[:, 0] * np.cos(polar[:, 1])
    y = polar[:, 0] * np.sin(polar[:, 1])
    x += centerX
    y += centerY
    return np.array([x, y]).T


photo3 = polar_to_photo(polar3, 1053, 1063)
sns.scatterplot(x=photo_original[:, 0], y=photo_original[:, 1])
sns.scatterplot(x=photo3[:, 0], y=photo3[:, 1])

In [ ]:
error3_1 = photo_original - photo3
sns.jointplot(x=error3_1[:, 0], y=error3_1[:, 1])
plt.xlabel('x error')
plt.ylabel('y error')
print(error3_1.min(axis=0), error3_1.max(axis=0))
print('mean (rho,theta)', np.abs(error3_1).mean(axis=0))

## Overall Curve Fit with polar_to_photo


In [ ]:
def overall_fit(azalt, scale, north_bias,
                k0, k1, k2, k3, k4, r0, r1, r2, t0, t1, t2, t3,
                r10, r11, r12, t10, t11, t12, t13,
                cx, cy):
    rho = np.deg2rad(90 - azalt[:, 1]) / (np.pi / 2)
    rho *= scale
    theta = np.deg2rad(azalt[:, 0])
    theta += north_bias  # np.deg2rad(155.6)=2.716  # initial rotation angle
    theta %= 2 * np.pi

    polar = np.array([rho, theta]).T

    rho1 = rho_projection_k(polar[:, 0], k0, k1, k2, k3, k4)
    rho2 = distortion_lm_rho(polar, r0, r1, r2, t0, t1, t2, t3)
    rho = rho1 + rho2

    theta = distortion_lm_theta(polar, r10, r11, r12, t10, t11, t12, t13)

    # transform to photo
    x = rho * np.cos(theta)
    y = rho * np.sin(theta)

    x += cx
    y += cy

    result = np.array([x, y]).T

    return result.flatten()


### Use parameters directly

directly use rho_pram, theta_pram
fitting parameters only scale, north_bias, cx, cy → central_param

In [ ]:
def overall_with_param(azalt, scale, north_bias, cx, cy):
    return overall_fit(azalt, scale, north_bias, *rho_pram, *theta_pram, cx, cy)


popt, pcov = curve_fit(overall_with_param,
                       xdata=azalt_original,
                       ydata=photo_original.flatten(),
                       p0=[983, np.deg2rad(155.6), 1053, 1063])

In [ ]:
central_param = popt
central_param


In [ ]:
photo_pred1 = overall_with_param(azalt_original, *central_param)
photo_pred1 = photo_pred1.reshape(-1, 2)

sns.scatterplot(x=photo_original[:, 0], y=photo_original[:, 1])
sns.scatterplot(x=photo_pred1[:, 0], y=photo_pred1[:, 1])

In [ ]:
error4 = photo_original - photo_pred1
sns.jointplot(x=error4[:, 0], y=error4[:, 1])
plt.xlabel('x error')
plt.ylabel('y error')
print(error4.min(axis=0), error4.max(axis=0), np.abs(error4).mean(axis=0))

error4_1 = np.sqrt((error4 ** 2).sum(axis=1))
print('mean', error4_1.mean(), 'median', np.median(error4_1))


### Use the previously mentioned parameters as initial values for fitting

set rho_pram, theta_pram as initial value `p0`, solve all parameters

```
scale, north_bias,
k0, k1, k2, k3, k4, r0, r1, r2, t0, t1, t2, t3,
r10, r11, r12, t10, t11, t12, t13,
cx, cy
```

In [ ]:

# test
azalt = np.array([[180, 50],
                  [180, 60],
                  [30, 80]])

overall_fit(azalt, 983, 0, *rho_pram, *theta_pram, 1053, 1063)


In [ ]:
popt, pcov = curve_fit(overall_fit, xdata=azalt_original, ydata=photo_original.flatten(),
                       p0=[983, np.deg2rad(155.6), *rho_pram, *theta_pram, 1053, 1063])

In [ ]:
full_param = popt
full_param

```python
full_param=array([ 9.80468512e+02,  2.72026459e+00,  2.29140748e-02, -1.08295168e-07,
        3.02312007e-13, -5.68179315e-19,  3.63150573e-25,  2.07858116e-03,
       -1.23195002e-08,  1.48715717e-14, -3.57886036e-01, -2.83555694e-01,
        8.36964518e-01,  1.68527286e-02,  7.36218690e-04, -1.76977466e-09,
        1.90190984e-15, -2.57021795e-03,  3.89713149e-03, -8.45247990e-04,
       -1.07870236e-04,  1.05506572e+03,  1.06466078e+03])
```

In [ ]:
photo_pred2 = overall_fit(azalt_original, *popt)

photo_pred2 = photo_pred2.reshape(-1, 2)

sns.scatterplot(x=photo_original[:, 0], y=photo_original[:, 1])
sns.scatterplot(x=photo_pred2[:, 0], y=photo_pred2[:, 1])

In [ ]:
error6 = photo_original - photo_pred2
sns.jointplot(x=error6[:, 0], y=error6[:, 1])
plt.xlabel('x error')
plt.ylabel('y error')
print(error6.min(axis=0), error6.max(axis=0), np.abs(error6).mean(axis=0))

error6_1 = np.sqrt((error6 ** 2).sum(axis=1))
print('mean', error6_1.mean(), 'median', np.median(error6_1))

## Use neural network to promote accuracy


In [ ]:
import torch


class Net(torch.nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.encoder = torch.nn.Sequential(
            torch.nn.Linear(2, 1024),
            torch.nn.Linear(1024, 2),
            torch.nn.Hardtanh(min_val=-0.1, max_val=0.1),
        )

    def forward(self, x):
        return self.encoder(x) + x

    def reset_parameters(self):
        for p in self.parameters():
            if p.dim() > 1:
                torch.nn.init.xavier_uniform_(p)
            else:
                torch.nn.init.zeros_(p)


net = Net()

# data
data = torch.tensor(photo_pred2, dtype=torch.float32)
tags = torch.tensor(photo_original, dtype=torch.float32)

if not os.path.exists('precisionUP.pth'):

    optimizer = torch.optim.Adam(net.parameters(), lr=0.01)
    criterion = torch.nn.MSELoss()

    epoch = 0
    while epoch < 10000:
        optimizer.zero_grad()

        pred = net(data)
        loss = criterion(pred, tags)

        # if epoch == 0 and loss.item() > 1e-7:
        #     net.reset_parameters()
        #     optimizer = torch.optim.Adam(net.parameters(), lr=0.01)
        #     print('reset parameters', loss.item())
        #     continue

        loss.backward()
        optimizer.step()

        pixel_loss = np.sqrt(loss.item())
        pixel_error = np.sqrt(((pred.detach().numpy() - tags.detach().numpy()) ** 2).sum(axis=1)).mean()
        if epoch % 2000 == 0:
            print(epoch, loss.item(), pixel_error)

        if pixel_error < 0.42:
            print(epoch, loss.item(), pixel_error)
            break

        epoch += 1
else:
    net = torch.load('precisionUP.pth')

In [ ]:
photo_pred3 = net(data).detach().numpy()

sns.scatterplot(x=photo_original[:, 0], y=photo_original[:, 1])
sns.scatterplot(x=photo_pred3[:, 0], y=photo_pred3[:, 1])

In [ ]:
error5 = photo_original - photo_pred3
sns.jointplot(x=error5[:, 0], y=error5[:, 1])
plt.xlabel('x error')
plt.ylabel('y error')
print(error5.min(axis=0), error5.max(axis=0), np.abs(error5).mean(axis=0))

error5_1 = np.sqrt((error5 ** 2).sum(axis=1))
print('mean', error5_1.mean(), 'median', np.median(error5_1))


In [ ]:
torch.save(net, 'precisionUP.pth')

# Save the model results

index:

cal_x, cal_y,real_x, real_y,  error_x, error_y, cal_rho, cal_theta, real_rho, real_theta, error_rho, error_theta

In [ ]:
rho_pram, theta_pram, central_param, full_param


```
rho_param = [ 2.01359497e-02, -1.09365603e-07,  3.27957013e-13, -6.51174844e-19,
          4.40910185e-25, -3.51667070e-03,  9.55189822e-09, -9.95838916e-15,
         -2.49109674e+00, -1.99486651e+00, -1.40387481e-02, -3.60503376e-02]
theta_param = [-3.29423016e-04,  1.32060166e-09, -1.36057949e-15, -9.30391856e-02,
         9.78303292e-02,  2.60579217e-02, -9.55923229e-03]

central_param = [ 982.98764146,    2.7200077 , 1053.12453008, 1063.02653908]

full_param = [ 9.80468512e+02,  2.72026459e+00,  2.29140748e-02, -1.08295168e-07,
         3.02312007e-13, -5.68179315e-19,  3.63150573e-25,  2.07858116e-03,
        -1.23195002e-08,  1.48715717e-14, -3.57886036e-01, -2.83555694e-01,
         8.36964518e-01,  1.68527286e-02,  7.36218690e-04, -1.76977466e-09,
         1.90190984e-15, -2.57021795e-03,  3.89713149e-03, -8.45247990e-04,
        -1.07870236e-04,  1.05506572e+03,  1.06466078e+03]

```

## polar1's result

In [ ]:
df_result1 = pd.DataFrame()
df_result1['cal_x'] = polar1[:, 0] * np.cos(polar1[:, 1])
df_result1['cal_y'] = polar1[:, 0] * np.sin(polar1[:, 1])
df_result1['real_x'] = photo_original[:, 0]
df_result1['real_y'] = photo_original[:, 1]
df_result1['error_x'] = df_result1['cal_x'] - df_result1['real_x']
df_result1['error_y'] = df_result1['cal_y'] - df_result1['real_y']
df_result1['cal_rho'] = polar1[:, 0]
df_result1['cal_theta'] = polar1[:, 1]
df_result1['real_rho'] = photo_polar[:, 0]
df_result1['real_theta'] = photo_polar[:, 1]
df_result1['error_rho'] = error1[:, 0]
df_result1['error_theta'] = error1[:, 1]

df_result1.to_csv('result.polar_projection.csv', index=False)
del df_result1

## polar2's result
radial and tangential distortion

In [ ]:
df_result2 = pd.DataFrame()
df_result2['cal_x'] = polar2[:, 0] * np.cos(polar2[:, 1])
df_result2['cal_y'] = polar2[:, 0] * np.sin(polar2[:, 1])
df_result2['real_x'] = photo_original[:, 0]
df_result2['real_y'] = photo_original[:, 1]
df_result2['error_x'] = df_result2['cal_x'] - df_result2['real_x']
df_result2['error_y'] = df_result2['cal_y'] - df_result2['real_y']
df_result2['cal_rho'] = polar2[:, 0]
df_result2['cal_theta'] = polar2[:, 1]
df_result2['real_rho'] = photo_polar[:, 0]
df_result2['real_theta'] = photo_polar[:, 1]
df_result2['error_rho'] = error2[:, 0]
df_result2['error_theta'] = error2[:, 1]

df_result2.to_csv('result.distortion.csv', index=False)
del df_result2

## polar3's result


projection + radial and tangential distortion


In [ ]:
df_result3 = pd.DataFrame()
df_result3['cal_x'] = polar3[:, 0] * np.cos(polar3[:, 1])
df_result3['cal_y'] = polar3[:, 0] * np.sin(polar3[:, 1])
df_result3['real_x'] = photo_original[:, 0]
df_result3['real_y'] = photo_original[:, 1]
df_result3['error_x'] = df_result3['cal_x'] - df_result3['real_x']
df_result3['error_y'] = df_result3['cal_y'] - df_result3['real_y']
df_result3['cal_rho'] = polar3[:, 0]
df_result3['cal_theta'] = polar3[:, 1]
df_result3['real_rho'] = photo_polar[:, 0]
df_result3['real_theta'] = photo_polar[:, 1]
df_result3['error_rho'] = error3[:, 0]
df_result3['error_theta'] = error3[:, 1]

df_result3.to_csv('result.overall.csv', index=False)
del df_result3

## photo_pred1's result



In [ ]:
df_result4 = pd.DataFrame()
df_result4['cal_x'] = photo_pred1[:, 0]
df_result4['cal_y'] = photo_pred1[:, 1]
df_result4['real_x'] = photo_original[:, 0]
df_result4['real_y'] = photo_original[:, 1]
df_result4['error_x'] = df_result4['cal_x'] - df_result4['real_x']
df_result4['error_y'] = df_result4['cal_y'] - df_result4['real_y']

df_result4.to_csv('result.overall_photo.csv', index=False)
del df_result4

## photo_pred2's result


In [ ]:
df_result5 = pd.DataFrame()
df_result5['cal_x'] = photo_pred2[:, 0]
df_result5['cal_y'] = photo_pred2[:, 1]
df_result5['real_x'] = photo_original[:, 0]
df_result5['real_y'] = photo_original[:, 1]
df_result5['error_x'] = df_result5['cal_x'] - df_result5['real_x']
df_result5['error_y'] = df_result5['cal_y'] - df_result5['real_y']

df_result5.to_csv('result.overall_photo2.csv', index=False)
del df_result5

## photo_pred3's result

In [ ]:
df_result6 = pd.DataFrame()
df_result6['cal_x'] = photo_pred3[:, 0]
df_result6['cal_y'] = photo_pred3[:, 1]
df_result6['real_x'] = photo_original[:, 0]
df_result6['real_y'] = photo_original[:, 1]
df_result6['error_x'] = df_result6['cal_x'] - df_result6['real_x']
df_result6['error_y'] = df_result6['cal_y'] - df_result6['real_y']

df_result6.to_csv('result.overall_photo3.csv', index=False)
del df_result6

 # Draw the altitude angle alt of each pixel in the photo coordinate system
 

In [ ]:
AZ, ALT = np.meshgrid(np.arange(0, 361, 2), np.arange(0, 91, 0.5))
AZALT = np.array([AZ.flatten(), ALT.flatten()]).T

photo_pred4 = overall_with_param(AZALT, *central_param)
photo_pred4 = photo_pred4.reshape(-1, 2)

photo_pred4 = net(torch.tensor(photo_pred4, dtype=torch.float32)).detach().numpy()


In [ ]:
photo_pred4

In [ ]:
from scipy.interpolate import griddata

grid_x, grid_y = np.mgrid[0:2080, 0:2080]

# points = photo_pred4
# values = AZALT[...,1]


gz = griddata(photo_pred4, AZALT[..., 1], (grid_x, grid_y), method='linear')

plt.imshow(gz, cmap='jet', vmin=0, vmax=90)
plt.colorbar()

In [ ]:
photo_pred5 = overall_fit(AZALT, *full_param)
photo_pred5 = photo_pred5.reshape(-1, 2)

gz = griddata(photo_pred5, AZALT[..., 1], (grid_y, grid_x), method='linear')

plt.imshow(gz, cmap='jet', vmin=0, vmax=90)
plt.colorbar()
plt.savefig('calibration.final.altitude.png')
plt.savefig('calibration.final.altitude.eps')

In [ ]:
gz = griddata(photo_pred5, AZALT[..., 0], (grid_y, grid_x), method='linear')
plt.imshow(gz, cmap='jet', vmin=0)
plt.colorbar()